In [2]:
import pandas as pd
import numpy as np
import re

import plotly.express as px

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

In [3]:
df = pd.read_csv("combined_pc_hardware_cleaned.csv")

df.head()

,Unnamed: 0,CPU,category,GPU,motherBoard,Ram,SSD,PowerSupply,cabinates,price
0,0.0,amd Ryzen 5 3600 with Wraith Stealth Cooler (1...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,9800.0
1,1.0,amd Ryzen 9 5900X 3.7 GHz Upto 4.8 GHz AM4 Soc...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,34890.0
2,2.0,processsor Ultra 3.5 GHz LGA 1150 Intel Core i...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,760.0
3,3.0,GIGASTAR 3.4 GHz LGA 1155 Intel i5-3570K For H...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,1890.0
4,4.0,Intel i5-12400F 4.4 GHz Upto 4.4 GHz LGA1700 S...,CPU,NaN,NaN,NaN,NaN,NaN,NaN,13008.0


In [4]:
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

In [5]:
df["product_name"] = (
    df["CPU"]
    .fillna(df["GPU"])
    .fillna(df["motherBoard"])
    .fillna(df["Ram"])
    .fillna(df["SSD"])
    .fillna(df["PowerSupply"])
    .fillna(df["cabinates"])
)

df = df.dropna(subset=["product_name"])

df = df.reset_index(drop=True)

In [6]:
import re

# Extract brand
df["brand"] = df["product_name"].str.extract(
    r'(Intel|AMD|Corsair|Kingston|Samsung|Gigabyte|ASUS|MSI|Crucial)',
    expand=False
)

# Extract GHz
df["clock_speed"] = df["product_name"].str.extract(r'(\d+\.\d+)\s*GHz')

# Extract RAM/Storage size
df["memory"] = df["product_name"].str.extract(r'(\d+)\s*GB')

# Extract TB storage
df["storage_tb"] = df["product_name"].str.extract(r'(\d+)\s*TB')

# Convert numeric
df["clock_speed"] = pd.to_numeric(df["clock_speed"], errors="coerce")
df["memory"] = pd.to_numeric(df["memory"], errors="coerce")
df["storage_tb"] = pd.to_numeric(df["storage_tb"], errors="coerce")

df["clock_speed"] = df["clock_speed"].fillna(0)
df["memory"] = df["memory"].fillna(0)
df["storage_tb"] = df["storage_tb"].fillna(0)

In [7]:
from sklearn.preprocessing import LabelEncoder

brand_encoder = LabelEncoder()

df["brand"] = df["brand"].fillna("Unknown")

df["brand_encoded"] = brand_encoder.fit_transform(df["brand"])

In [8]:
# Fix missing numeric values

df["price"] = df["price"].fillna(df["price"].median())

df["clock_speed"] = df["clock_speed"].fillna(0)

df["memory"] = df["memory"].fillna(0)

df["storage_tb"] = df["storage_tb"].fillna(0)

df["brand_encoded"] = df["brand_encoded"].fillna(0)

In [9]:
from sklearn.preprocessing import LabelEncoder

# encode target variable
encoder = LabelEncoder()

df["category_encoded"] = encoder.fit_transform(df["category"])

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=1505,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df["product_name"])

y = df["category_encoded"]

In [11]:
from scipy.sparse import hstack
import numpy as np

numeric_features = df[[
    "price",
    "brand_encoded",
    "clock_speed",
    "memory",
    "storage_tb"
]].astype(float)

numeric_features = numeric_features.fillna(0)

X = hstack([X_text, numeric_features.values])

y = df["category_encoded"]

NameError: name 'X_text' is not defined

In [517]:
df["product_name"] = (
    df["CPU"]
    .fillna(df["GPU"])
    .fillna(df["motherBoard"])
    .fillna(df["Ram"])
    .fillna(df["SSD"])
    .fillna(df["PowerSupply"])
    .fillna(df["cabinates"])
)

df = df.dropna(subset=["product_name"])

df["product_name"] = df["product_name"].astype(str)

In [518]:
print("Any NaN in numeric features:", numeric_features.isna().sum())

Any NaN in numeric features: price            0
brand_encoded    0
clock_speed      0
memory           0
storage_tb       0
dtype: int64


In [519]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [520]:
print(df.isnull().sum())

CPU                 10360
category                0
GPU                 11200
motherBoard         10360
Ram                 10360
SSD                  9160
PowerSupply         10680
cabinates            8680
price                   0
product_name            0
brand                   0
clock_speed             0
memory                  0
storage_tb              0
brand_encoded           0
category_encoded        0
dtype: int64


In [521]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=320,
    max_depth=22,
    min_samples_split=3,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,n_estimators,320
,criterion,'gini'
,max_depth,22
,min_samples_split,3
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [522]:
y_pred = model.predict(X_test)

In [523]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=3000,
    C=4
)

model.fit(X_train, y_train)

c:\Users\nagul\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning:

lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



,penalty,'l2'
,dual,False
,tol,0.0001
,C,4
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,3000
,multi_class,'deprecated'


In [524]:
import joblib

print("TFIDF feature size:", X.shape[1])

joblib.dump(model, "hardware_classifier.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

TFIDF feature size: 1505


['tfidf_vectorizer.pkl']

In [525]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)
print(classification_report(y_test, y_pred))

Model Accuracy: 0.9385593220338984
              precision    recall  f1-score   support

           0       0.98      0.89      0.93       288
           1       0.98      0.90      0.94       120
           2       0.87      0.81      0.84       288
           3       1.00      0.92      0.96       224
           4       0.93      0.99      0.96       288
           5       0.98      0.99      0.99       528
           6       0.90      0.97      0.93       624

    accuracy                           0.94      2360
   macro avg       0.95      0.92      0.93      2360
weighted avg       0.94      0.94      0.94      2360



In [526]:
fig = px.histogram(
    df,
    x="category",
    title="Hardware Category Distribution",
    color="category"
)

fig.show()

In [527]:
fig = px.histogram(
    df,
    x="price",
    nbins=50,
    title="Hardware Price Distribution"
)

fig.show()

In [528]:
features = ["price"]

X = df[features]

y = df["category_encoded"]

In [529]:
import joblib

model = joblib.load("hardware_classifier.pkl")

print("Model expects features:", model.n_features_in_)

Model expects features: 1505


In [530]:
tfidf = joblib.load("tfidf_vectorizer.pkl")

print("TFIDF features:", len(tfidf.get_feature_names_out()))

TFIDF features: 1505


In [533]:
import joblib

model = joblib.load("hardware_classifier.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")

print("Model expects features:", model.n_features_in_)
print("TFIDF feature size:", len(tfidf.get_feature_names_out()))

Model expects features: 1505
TFIDF feature size: 1505


In [534]:
print("TFIDF feature size:", X.shape[1])

TFIDF feature size: 1


In [535]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Load dataset
df = pd.read_csv("combined_pc_hardware_cleaned.csv")

# Create product name column
df["product_name"] = (
    df["CPU"]
    .fillna(df["GPU"])
    .fillna(df["motherBoard"])
    .fillna(df["Ram"])
    .fillna(df["SSD"])
    .fillna(df["PowerSupply"])
    .fillna(df["cabinates"])
)

# Drop missing
df = df.dropna(subset=["product_name"])

# TF-IDF
tfidf_rec = TfidfVectorizer(stop_words="english")

X = tfidf_rec.fit_transform(df["product_name"])

# KNN model
knn_model = NearestNeighbors(
    n_neighbors=6,
    metric="cosine"
)

knn_model.fit(X)

# Save files
joblib.dump(knn_model, "knn_recommender.pkl")
joblib.dump(tfidf_rec, "tfidf_recommender.pkl")
joblib.dump(df, "hardware_dataset.pkl")

print("Recommendation model saved!")

Recommendation model saved!


In [538]:
from sklearn.preprocessing import LabelEncoder
import joblib

# Create encoder
encoder = LabelEncoder()

# Fit encoder on category column
df["category_encoded"] = encoder.fit_transform(df["category"])

# Save encoder
joblib.dump(encoder, "category_encoder.pkl")

print("Encoder saved successfully")

Encoder saved successfully


In [540]:
encoder = joblib.load("category_encoder.pkl")

In [544]:
df = df.drop_duplicates(subset="product_name")

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11800 entries, 0 to 11799
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CPU               1440 non-null   object 
 1   category          11800 non-null  object 
 2   GPU               600 non-null    object 
 3   motherBoard       1440 non-null   object 
 4   Ram               1440 non-null   object 
 5   SSD               2640 non-null   object 
 6   PowerSupply       1120 non-null   object 
 7   cabinates         3120 non-null   object 
 8   price             11800 non-null  float64
 9   product_name      11800 non-null  object 
 10  brand             11800 non-null  object 
 11  clock_speed       11800 non-null  float64
 12  memory            11800 non-null  float64
 13  storage_tb        11800 non-null  float64
 14  brand_encoded     11800 non-null  int64  
 15  category_encoded  11800 non-null  int64  
dtypes: float64(4), int64(2), object(10)
memo

In [14]:
fig = px.box(
    df,
    x="category",
    y="price",
    title="Price Distribution Across Hardware Categories",
    color="category"
)

fig.show()